# Vistazos a la base de datos

In [ ]:
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
import pandas as pd
import numpy_financial as npf

from src.utils import select_file

# path = select_file()
df = pd.read_excel(path, index_col="ID")
df["TEM"] = df.apply(
    lambda row: npf.rate(
        nper=row["PLAZO"], pmt=-row["CUOTA A PAGAR"], pv=row["CAPITAL"], fv=0
    ),
    axis=1,
)
df["FECHA"] = pd.to_datetime(df["FECHA"], dayfirst=False).dt.to_period("D")
df = df.loc[df.index != "Total"]
df

In [ ]:
cuotas = []
for id in df.index:
    tem = df.at[id, "TEM"]
    cap = df.at[id, "CAPITAL"]
    plazo = df.at[id, "PLAZO"]
    emisión = pd.Period(df.at[id, "FECHA"], freq="M")
    val_cta = df.at[id, "CUOTA A PAGAR"]
    for j in range(1, plazo+1):
        vto = emisión + j + 1
        vto = pd.Period(year=vto.year, month=vto.month, day=28, freq="D")
        cap_j = round(npf.ppmt(tem, j, plazo, -cap), 2)
        int_j = round((val_cta - cap_j)/1.21, 2)
        cuotas.append({
            "Crédito": id,
            "Cuota": j,
            "Vencimiento": vto, 
            "Capital": cap_j,
            "Interés": int_j,
            "IVA": val_cta - cap_j - int_j,
            "Total": val_cta
            })
        i += 1

df_ctas = pd.DataFrame(cuotas)
df_ctas.set_index(["Crédito", "Cuota", "Vencimiento"], inplace=True)

with pd.ExcelWriter(path) as writer:
    df.to_excel(writer, sheet_name="Créditos", index=True)
    df_ctas.reset_index().to_excel(writer, sheet_name="Cuotas", index=False)